#### Single Class Modulation per Channel

In [ ]:
%matplotlib qt
%load_ext autoreload
%autoreload 2
from src.plot import plot_population_heatmaps
subject = "Wifi"
session = "Wifi_20210618"
band = "gamma"

plot_population_heatmaps(subject, session, band, order=False)

In [ ]:
%matplotlib qt
%load_ext autoreload
%autoreload 2
from src.plot import plot_population_heatmaps

subject = "Router"
session = "Router_20220211"
band = "beta"

plot_population_heatmaps(subject, session, band, order=False)

#### Selectivity Analysis (Mixed ANOVA and Post-Hoc Comparison)

Check single-channel single-band modulation with rmANOVA

In [ ]:
%load_ext autoreload
%autoreload 2
from src.config import INTERIM_DATA_DIR

from src.statistical_analysis import analyze_event_modulation

# Iterate over all subjects in the interim directory
for subject_dir in INTERIM_DATA_DIR.iterdir():
    if not subject_dir.is_dir():
        continue
        
    subject = subject_dir.name
    
    # Iterate over all sessions for the current subject
    for session_dir in subject_dir.iterdir():
        if not session_dir.is_dir():
            continue
        session = session_dir.name        
        analyze_event_modulation(subject=subject, session=session)

Check mixed-selectivity with 2-ways Mixed-ANOVA

In [ ]:
%load_ext autoreload
%autoreload 2
from src.config import INTERIM_DATA_DIR

from src.statistical_analysis import analyze_selectivity

# Iterate over all subjects in the interim directory
for subject_dir in INTERIM_DATA_DIR.iterdir():
    if not subject_dir.is_dir():
        continue
        
    subject = subject_dir.name
    
    # Iterate over all sessions for the current subject
    for session_dir in subject_dir.iterdir():
        if not session_dir.is_dir():
            continue
            
        session = session_dir.name
                    
        print(f"Extracting informative channels for -> Subject: {subject} | Session: {session} | ALL EVENTS")
        
        analyze_selectivity(subject=subject, session=session)

In [16]:
import pandas as pd
from collections import Counter
from pathlib import Path

def summarize_selectivity_results(csv_filepath: str | Path, alpha: float = 0.05) -> None:
    """
    Reads the output CSV from analyze_selectivity and prints a summary 
    of the statistical findings per frequency band.
    """
    # Load dataset
    try:
        df = pd.read_csv(csv_filepath)
    except Exception as e:
        print(f"[ERROR] Could not read file {csv_filepath}: {e}")
        return
        
    if df.empty:
        print("[WARNING] The provided CSV is empty.")
        return

    print(f"--- SELECTIVITY RESULTS SUMMARY ---")
    print(f"File: {Path(csv_filepath).name}")
    
    bands = df['Band'].unique()
    
    # Iterate through bands to calculate and print summaries
    for band in bands:
        print(f"\n{'='*60}")
        print(f"  {band.upper()} BAND")
        print(f"{'='*60}")
        
        df_band = df[df['Band'] == band]
        total_channels = len(df_band)
        
        # Calculate key metrics
        sig_interaction = (df_band['p_interaction_fdr'] < alpha).sum()
        sig_main_bin = (df_band['p_main_bin_fdr'] < alpha).sum()
        
        print(f"  Total Channels: {total_channels}")
        print(f"  Main effect 'Bin' significant (FDR < {alpha}): {sig_main_bin}")
        print(f"  Interaction significant (FDR < {alpha}): {sig_interaction}")
        print(f"  Categories:")
        
        # Aggregate category counts
        cats = Counter(df_band['category'])
        for cat, count in sorted(cats.items()):
            print(f"    - {cat}: {count}")

csv_filepath = Path(r"C:\Users\tommy\OneDrive - Scuola Superiore Sant'Anna\Monkeys Parma\processed\Router\Router_20220211\selectivity_results.csv")
summarize_selectivity_results(csv_filepath)

--- SELECTIVITY RESULTS SUMMARY ---
File: selectivity_results.csv

  DELTA BAND
  Total Channels: 128
  Main effect 'Bin' significant (FDR < 0.05): 69
  Interaction significant (FDR < 0.05): 14
  Categories:
    - hook_specific: 1
    - mixed_steps_floor_diff: 5
    - mixed_steps_hook_diff: 2
    - motor_aspecific: 58
    - motor_specific: 5
    - non_informative: 56
    - steps_specific: 1

  THETA BAND
  Total Channels: 128
  Main effect 'Bin' significant (FDR < 0.05): 66
  Interaction significant (FDR < 0.05): 25
  Categories:
    - floor_specific: 9
    - hook_specific: 4
    - mixed_steps_floor_diff: 3
    - motor_aspecific: 49
    - motor_specific: 7
    - non_informative: 54
    - steps_specific: 2

  ALPHA BAND
  Total Channels: 128
  Main effect 'Bin' significant (FDR < 0.05): 49
  Interaction significant (FDR < 0.05): 8
  Categories:
    - floor_specific: 1
    - hook_specific: 2
    - mixed_steps_floor_diff: 1
    - motor_aspecific: 46
    - motor_specific: 4
    - non_infor